In [8]:
import pandas as pd
import numpy as np

train = pd.read_csv('train_features.csv', dtype={'StateHoliday': str}, parse_dates=['Date'])

test = pd.read_csv('test_features.csv', dtype={'StateHoliday': str}, parse_dates=['Date'])

In [9]:
FEATURES = [
    "DayOfWeek", "Month", "WeekOfYear", "IsWeekend",
    "Promo", "StateHoliday_code", "SchoolHoliday",
    "Segment_code", "CompetitionDistance_log", "CompetitionDistance_missing",
    "competition_open_date_known", "Promo2_enrolled", "Promo2_round_start_month",
    "lag_49", "lag_56", "roll_mean_28", "lag_49_filled", "lag_56_filled",
]

BANNED = ["Customers", "Sales", "Store", "Date", "Year", "Open"]

In [10]:
print(train.shape, test.shape)

print("train:", train.Date.min().date(), train.Date.max().date())

print("test :", test.Date.min().date(), test.Date.max().date(), test.Date.nunique(), "days")

(1017209, 37) (41088, 36)
train: 2013-01-01 2015-07-31
test : 2015-08-01 2015-09-17 48 days


In [11]:
assert len(set(FEATURES)) == 18

assert not set(FEATURES) & set(BANNED)

assert set(FEATURES) <= set(train) and set(FEATURES) <= set(test)

assert train[FEATURES].notna().all().all() and test[FEATURES].notna().all().all()

assert not train.duplicated(["Store", "Date"]).any()

assert test.Date.nunique() == 48 and train.Date.max() < test.Date.min()

In [12]:
print(train.Open.value_counts().to_dict())

{1: 844392, 0: 172817}


In [13]:
print("stores in Jul-Dec 2014:", train[train.Date.between("2014-07-01", "2014-12-31")].Store.nunique())

stores in Jul-Dec 2014: 935


In [15]:
full_idx = pd.date_range(train.Date.min(), test.Date.max())

full_idx

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-09-08', '2015-09-09', '2015-09-10', '2015-09-11',
               '2015-09-12', '2015-09-13', '2015-09-14', '2015-09-15',
               '2015-09-16', '2015-09-17'],
              dtype='datetime64[ns]', length=990, freq='D')

In [16]:
wide = (train.assign(S=train.Sales.where(train.Open == 1))
        .pivot(index="Date", columns="Store", values="S").reindex(full_idx))

raw = wide.shift(48).rolling(28, min_periods=1).mean()

raw

Store,1,2,3,4,5,6,7,8,9,10,...,1106,1107,1108,1109,1110,1111,1112,1113,1114,1115
2013-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-09-13,4455.208333,4996.833333,6792.958333,10202.916667,4544.916667,4479.541667,10061.541667,6424.541667,7182.958333,6093.458333,...,5119.458333,6076.125000,5900.291667,4731.958333,4454.083333,4514.041667,8356.916667,6702.416667,23043.375000,6889.916667
2015-09-14,4425.041667,4822.166667,6609.750000,10084.958333,4521.916667,4371.708333,9957.541667,6294.583333,7112.291667,5992.416667,...,5027.000000,5936.208333,5822.625000,4672.916667,4402.750000,4411.208333,8258.958333,6653.791667,22868.541667,6868.375000
2015-09-15,4406.666667,4820.958333,6580.541667,10055.708333,4507.166667,4349.666667,10001.291667,6232.333333,7082.541667,5893.583333,...,5032.208333,6037.458333,5789.958333,4752.833333,4319.250000,4461.458333,8198.916667,6587.875000,22817.250000,6866.708333
2015-09-16,4384.250000,4795.041667,6644.166667,10058.375000,4461.666667,4326.958333,10024.791667,6222.416667,7046.875000,5888.125000,...,5044.875000,6143.541667,5782.541667,4803.125000,4262.166667,4516.375000,8290.791667,6569.500000,22796.291667,6931.166667


In [17]:
r = raw.rename_axis("Date").reset_index().melt("Date", var_name="Store", value_name="roll_raw")

r

,Date,Store,roll_raw
0,2013-01-01,1,NaN
1,2013-01-02,1,NaN
2,2013-01-03,1,NaN
3,2013-01-04,1,NaN
4,2013-01-05,1,NaN
...,...,...,...
1103845,2015-09-13,1115,6889.916667
1103846,2015-09-14,1115,6868.375000
1103847,2015-09-15,1115,6866.708333
1103848,2015-09-16,1115,6931.166667


In [18]:
m = train[["Date", "Store", "roll_mean_28", "lag_49", "lag_49_filled"]].merge(r, on=["Date", "Store"], how="left")

assert len(m) == len(train)

ok = m.roll_raw.notna()

In [19]:
print("raw nulls:", (~ok).sum(), "| share:", round((~ok).mean(), 4))

raw nulls: 64087 | share: 0.063


In [20]:
print("non-null rows equal to stored feature:", np.isclose(m.roll_mean_28[ok], m.roll_raw[ok]).mean())

non-null rows equal to stored feature: 1.0


In [22]:
nul = m[~ok]

nul

,Date,Store,roll_mean_28,lag_49,lag_49_filled,roll_raw
89873,2015-05-12,674,5966.000000,5966.000000,0,NaN
90988,2015-05-11,674,7150.000000,7150.000000,0,NaN
92103,2015-05-10,674,8193.349866,8193.349866,1,NaN
93218,2015-05-09,674,2401.000000,2401.000000,0,NaN
94333,2015-05-08,674,7438.000000,7438.000000,0,NaN
...,...,...,...,...,...,...
1017204,2013-01-01,1111,5251.702182,5251.702182,1,NaN
1017205,2013-01-01,1112,10236.577664,10236.577664,1,NaN
1017206,2013-01-01,1113,6627.859694,6627.859694,1,NaN
1017207,2013-01-01,1114,20666.562500,20666.562500,1,NaN


In [23]:
cold = nul.Date < train.Date.min() + pd.Timedelta(days=48)

In [24]:
gap = nul.Date.between("2015-01-01", "2015-02-17") & ~cold

other = ~cold & ~gap

In [25]:
print("cold start:", cold.sum(), "| gap windows:", gap.sum(), "| other:", other.sum())

print(nul[other].groupby("Store").size().sort_values(ascending=False).head(5))

cold start: 53519 | gap windows: 8640 | other: 1928
Store
1081    185
103     185
349     101
708      63
972      60
dtype: int64


In [26]:
print("stored value on null rows == lag_49:", np.isclose(nul.roll_mean_28, nul.lag_49).mean())

print("lag_49_filled on null rows:", nul.lag_49_filled.mean().round(3))

print(nul.roll_mean_28.round(3).value_counts().head(3))

stored value on null rows == lag_49: 1.0
lag_49_filled on null rows: 0.999
roll_mean_28
5924.919    233
4689.498    233
8159.216    149
Name: count, dtype: int64


# Project Status: DemandLens paused after Stage 3, Part A

**Status:** Paused. Stage 3, Part B (baseline modeling) was not completed.

## Completed
- **Stage 1:** Data cleaning and validation of the Rossmann train, test and store tables.
- **Stage 2:** Exploratory data analysis, covering promo effects, seasonality, store segments, holidays and closure patterns.
- **Stage 3, Part A:** Feature engineering. This produced an explicit 18-feature list, a 48-day forecast horizon, lag and rolling features of at least 49 days with no leakage found, and a documented list of excluded columns.
- All decisions are recorded in `decision_log.md`, and EDA findings are in `eda_findings.md`.

## Not completed
- Naive baseline, time-respecting validation folds, and the linear, random forest and gradient-boosted baselines.
- The Year and Open feature experiments.
- Everything in Stage 4 and later (advanced forecasting, error analysis, the GenAI advisor layer and the dashboard).

## Reason for pausing
Several feature-lineage questions were still open when Part B began. Reported baseline numbers would depend on these features, so I decided not to build a model on top of them until they were resolved:
- The cause of the 64,087 `roll_mean_28` nulls is inferred (cold start, plus the 180-store gap in Jul–Dec 2014) but not directly verified.
- The explanation for the earlier ~29.5k lag mismatch (NaN vs 0 handling) is unproven.
- The origin of the 4759.096 fill value is untraced.
- The training population (open days only vs all days) was still undecided.

These are recorded as **unverified assumptions**, not facts.

## If resumed
1. Trace the three open items above.
2. Decide the training population, then settle the `Open` feature.
3. Build a horizon-valid naive baseline and 3 or more time-respecting 48-day folds.
4. Train the linear, random forest and gradient-boosted baselines on identical folds, then run the Year experiment.